In [3]:
"""
咖啡店滿意度檢定 — 1D t-test vs 3D Hotelling T²
"""
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)

# ============================================================
# STEP 0: 造資料 (假裝這是 30 位顧客的回饋)
# ============================================================
n = 30
# 真實 mean 故意設定為 (7.5, 7.8, 7.6) — 略低於品牌標準 8
true_mu = np.array([7.5, 7.8, 7.6])
# 三個面向相關 (顧客打分有整體傾向)
true_Sigma = np.array([[1.0, 0.6, 0.5],
                       [0.6, 1.0, 0.5],
                       [0.5, 0.5, 1.0]])
data = rng.multivariate_normal(true_mu, true_Sigma, size=n)
# data shape: (30, 3),每列是 (coffee, service, ambiance)

print("資料前 3 筆:")
print(data[:3])
print(f"\n資料 shape: {data.shape}\n")


# ============================================================
# STEP 1: 算 sample 統計量
# ============================================================
x_bar = data.mean(axis=0)           # sample mean vector
S = np.cov(data, rowvar=False)      # sample covariance matrix (3x3)

print("=" * 55)
print("STEP 1: Sample 統計量")
print("=" * 55)
print(f"Sample mean x̄ = {x_bar}")
print(f"Sample covariance S =\n{S}")


# ============================================================
# STEP 2A: 1D t-test (只看「咖啡品質」這一個變數)
# ============================================================
print("\n" + "=" * 55)
print("STEP 2A: 1D t-test (只驗咖啡品質 = 8)")
print("=" * 55)

mu_0_1d = 8.0
x_coffee = data[:, 0]               # 取第 0 個 column (咖啡品質)

x_bar_1d = x_coffee.mean()
s_1d = x_coffee.std(ddof=1)
se_1d = s_1d / np.sqrt(n)
t_stat = (x_bar_1d - mu_0_1d) / se_1d
df_1d = n - 1
p_1d = 2 * (1 - stats.t.cdf(abs(t_stat), df=df_1d))

print(f"x̄  = {x_bar_1d:.4f}")
print(f"s   = {s_1d:.4f}")
print(f"SE  = s/√n = {se_1d:.4f}")
print(f"t   = (x̄ - μ₀) / SE = {t_stat:.4f}")
print(f"df  = {df_1d}")
print(f"p   = {p_1d:.4f}")
print(f"判定: {'拒絕 H₀' if p_1d < 0.05 else '不拒絕 H₀'}")

# scipy 驗證
_, p_scipy = stats.ttest_1samp(x_coffee, mu_0_1d)
print(f"\nscipy 驗證: p = {p_scipy:.4f} ✓")



資料前 3 筆:
[[7.28118469 6.90996074 7.99053818]
 [5.5647253  7.02943006 8.03980335]
 [7.29406341 7.60909082 7.69082836]]

資料 shape: (30, 3)

STEP 1: Sample 統計量
Sample mean x̄ = [7.502996   7.72208563 7.63411813]
Sample covariance S =
[[0.79060401 0.58129193 0.24178319]
 [0.58129193 0.69070846 0.36329962]
 [0.24178319 0.36329962 0.4687718 ]]

STEP 2A: 1D t-test (只驗咖啡品質 = 8)
x̄  = 7.5030
s   = 0.8892
SE  = s/√n = 0.1623
t   = (x̄ - μ₀) / SE = -3.0615
df  = 29
p   = 0.0047
判定: 拒絕 H₀

scipy 驗證: p = 0.0047 ✓


In [4]:

# ============================================================
# STEP 2B: Hotelling's T² (同時驗三個變數 = (8, 8, 8))
# ============================================================
print("\n" + "=" * 55)
print("STEP 2B: Hotelling's T² (同時驗三變數 = (8,8,8))")
print("=" * 55)

mu_0_3d = np.array([8.0, 8.0, 8.0])
p = 3                               # 變數個數

diff = x_bar - mu_0_3d
S_inv = np.linalg.inv(S)
T2 = n * diff @ S_inv @ diff
df1, df2 = p, n - p                 # T² 換算 F 用的自由度

# T²_{p, n-1} 沒有直接的 scipy 分布,但可以換算成 F:
#   (n-p)/((n-1)*p) * T² ~ F(p, n-p)
F_stat = (n - p) / ((n - 1) * p) * T2
p_3d = 1 - stats.f.cdf(F_stat, df1, df2)

print(f"x̄ - μ₀ = {diff}")
print(f"T² = n (x̄-μ₀)ᵀ S⁻¹ (x̄-μ₀) = {T2:.4f}")
print(f"\n換算到 F 分布:")
print(f"  F = ((n-p)/((n-1)p)) · T² = {F_stat:.4f}")
print(f"  df = ({df1}, {df2})")
print(f"  p-value = {p_3d:.4f}")
print(f"判定: {'拒絕 H₀' if p_3d < 0.05 else '不拒絕 H₀'}")





STEP 2B: Hotelling's T² (同時驗三變數 = (8,8,8))
x̄ - μ₀ = [-0.497004   -0.27791437 -0.36588187]
T² = n (x̄-μ₀)ᵀ S⁻¹ (x̄-μ₀) = 18.8699

換算到 F 分布:
  F = ((n-p)/((n-1)p)) · T² = 5.8562
  df = (3, 27)
  p-value = 0.0032
判定: 拒絕 H₀


In [5]:
# ============================================================
# STEP 3: 對比兩個檢定的結論
# ============================================================
print("\n" + "=" * 55)
print("STEP 3: 對比")
print("=" * 55)
print(f"{'檢定':<25} {'p-value':<12} {'結論'}")
print("-" * 55)
print(f"{'1D t-test (僅咖啡)':<22} {p_1d:<12.4f} {'拒絕' if p_1d<0.05 else '不拒絕'}")
print(f"{'3D Hotelling T² (全部)':<19} {p_3d:<12.4f} {'拒絕' if p_3d<0.05 else '不拒絕'}")
print("""
解讀: 
- 1D 只看咖啡品質 → 可能因為單一面向偏差不大,沒拒絕
- 3D 同時看三個面向 → 偏差「累積」起來,通常更容易拒絕
- 3D 還會考量變數間的相關 (用 S⁻¹), 抓得到 1D 抓不到的偏離

實務啟示: 當你關心「整體 mean vector」時,單獨對每個變數
跑 t-test 會錯失資訊 + 有 multiple testing 問題,Hotelling
T² 一次解決
""")


STEP 3: 對比
檢定                        p-value      結論
-------------------------------------------------------
1D t-test (僅咖啡)        0.0047       拒絕
3D Hotelling T² (全部) 0.0032       拒絕

解讀: 
- 1D 只看咖啡品質 → 可能因為單一面向偏差不大,沒拒絕
- 3D 同時看三個面向 → 偏差「累積」起來,通常更容易拒絕
- 3D 還會考量變數間的相關 (用 S⁻¹), 抓得到 1D 抓不到的偏離

實務啟示: 當你關心「整體 mean vector」時,單獨對每個變數
跑 t-test 會錯失資訊 + 有 multiple testing 問題,Hotelling
T² 一次解決

